In [11]:
"""
Data Preprocessing Pipeline for ADAM-Sense Dataset
Prepares data for model training
"""

import pandas as pd
import numpy as np
from pathlib import Path
import pickle

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    # Sensors
    'sensor_columns': ['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz'],
    'num_features': 6,
    
    # Activities
    'selected_activities': [
        'Eating',
        'Nail_Biting',
        'Face_Touch',
        'Smoking',
        'Staying_Still'
    ],
    
    # Windowing
    'sampling_rate': 50,  # Hz (estimated)
    'window_size': 128,  # samples (~2.56 seconds)
    'overlap': 0.5,  # 50%
    'step_size': 64,  # samples (window_size * (1 - overlap))
    
    # Train/test split - SIMPLE VERSION
    # Just specify which users go where
    'test_split_ratio': 0.2,  # 20% for testing
    'random_seed': 42,
}

print("="*60)
print("DATA PREPROCESSING PIPELINE")
print("="*60)

# ============================================================================
# STEP 1: Load Dataset
# ============================================================================
print("\n" + "="*60)
print("STEP 1: Loading Dataset")
print("="*60)

data_path = Path('../data/raw/StressSense.csv')
df = pd.read_csv(data_path)
print(f"✓ Loaded {len(df):,} samples")

# ============================================================================
# STEP 2: Filter Selected Activities
# ============================================================================
print("\n" + "="*60)
print("STEP 2: Filtering Activities")
print("="*60)

df_filtered = df[df['Activity'].isin(CONFIG['selected_activities'])].copy()
print(f"✓ Filtered to {len(df_filtered):,} samples")

print("\nSamples per activity:")
for activity in CONFIG['selected_activities']:
    count = len(df_filtered[df_filtered['Activity'] == activity])
    print(f"  {activity:20s}: {count:6,}")

# ============================================================================
# STEP 3: Auto-generate Train/Test Split
# ============================================================================
print("\n" + "="*60)
print("STEP 3: Creating Train/Test Split")
print("="*60)

# Get all unique users
all_users = sorted(df_filtered['User'].unique())
print(f"Total users found: {len(all_users)}")

# Random split
np.random.seed(CONFIG['random_seed'])
shuffled_users = np.array(all_users)
np.random.shuffle(shuffled_users)

n_test = max(3, int(len(all_users) * CONFIG['test_split_ratio']))
n_train = len(all_users) - n_test

train_users = sorted(shuffled_users[:n_train].tolist())
test_users = sorted(shuffled_users[n_train:].tolist())

print(f"\nTrain users ({n_train}): {train_users}")
print(f"Test users ({n_test}): {test_users}")

# Verify data split
train_samples = len(df_filtered[df_filtered['User'].isin(train_users)])
test_samples = len(df_filtered[df_filtered['User'].isin(test_users)])
print(f"\nTrain samples: {train_samples:,} ({train_samples/(train_samples+test_samples)*100:.1f}%)")
print(f"Test samples: {test_samples:,} ({test_samples/(train_samples+test_samples)*100:.1f}%)")

# ============================================================================
# STEP 4: Create Activity Label Mapping
# ============================================================================
print("\n" + "="*60)
print("STEP 4: Creating Label Mapping")
print("="*60)

activity_to_label = {activity: idx for idx, activity in enumerate(CONFIG['selected_activities'])}
label_to_activity = {idx: activity for activity, idx in activity_to_label.items()}

print("Label mapping:")
for activity, label in activity_to_label.items():
    print(f"  {label}: {activity}")

df_filtered['label'] = df_filtered['Activity'].map(activity_to_label)

# ============================================================================
# STEP 5: Create Sliding Windows
# ============================================================================
print("\n" + "="*60)
print("STEP 5: Creating Sliding Windows")
print("="*60)

def create_windows(data, labels, window_size, step_size):
    """Create sliding windows from continuous sensor data"""
    windows = []
    window_labels = []
    
    for start in range(0, len(data) - window_size + 1, step_size):
        end = start + window_size
        
        window = data[start:end, :]
        window_label_values = labels[start:end]
        
        # Get most common label (majority voting)
        unique, counts = np.unique(window_label_values, return_counts=True)
        most_common_label = unique[np.argmax(counts)]
        
        # Keep window if >80% same label
        if np.max(counts) / len(window_label_values) > 0.8:
            windows.append(window)
            window_labels.append(most_common_label)
    
    return np.array(windows), np.array(window_labels)

# Process data
all_train_windows = []
all_train_labels = []
all_test_windows = []
all_test_labels = []

sensor_cols = CONFIG['sensor_columns']

for user in all_users:
    user_data = df_filtered[df_filtered['User'] == user]
    
    for activity in CONFIG['selected_activities']:
        activity_data = user_data[user_data['Activity'] == activity]
        
        if len(activity_data) < CONFIG['window_size']:
            continue
        
        X = activity_data[sensor_cols].values
        y = activity_data['label'].values
        
        X_windows, y_windows = create_windows(
            X, y, 
            CONFIG['window_size'], 
            CONFIG['step_size']
        )
        
        if len(X_windows) == 0:
            continue
        
        # Assign to train or test
        if user in train_users:
            all_train_windows.append(X_windows)
            all_train_labels.append(y_windows)
        else:
            all_test_windows.append(X_windows)
            all_test_labels.append(y_windows)

# Combine all windows
X_train = np.concatenate(all_train_windows, axis=0)
y_train = np.concatenate(all_train_labels, axis=0)
X_test = np.concatenate(all_test_windows, axis=0)
y_test = np.concatenate(all_test_labels, axis=0)

print(f"✓ Created windows")
print(f"  Training: {X_train.shape[0]:,} windows")
print(f"  Test: {X_test.shape[0]:,} windows")

# ============================================================================
# STEP 6: Normalize Data
# ============================================================================
print("\n" + "="*60)
print("STEP 6: Normalizing Data")
print("="*60)

mean = X_train.mean(axis=(0, 1))
std = X_train.std(axis=(0, 1))

X_train_normalized = (X_train - mean) / std
X_test_normalized = (X_test - mean) / std

print(f"✓ Data normalized")

# ============================================================================
# STEP 7: Class Distribution
# ============================================================================
print("\n" + "="*60)
print("STEP 7: Class Distribution")
print("="*60)

print("Training set:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for label, count in zip(unique_train, counts_train):
    activity_name = label_to_activity[label]
    percentage = (count / len(y_train)) * 100
    print(f"  {label}: {activity_name:20s} - {count:6,} ({percentage:5.2f}%)")

print("\nTest set:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for label, count in zip(unique_test, counts_test):
    activity_name = label_to_activity[label]
    percentage = (count / len(y_test)) * 100
    print(f"  {label}: {activity_name:20s} - {count:6,} ({percentage:5.2f}%)")

# ============================================================================
# STEP 8: Save Preprocessed Data
# ============================================================================
print("\n" + "="*60)
print("STEP 8: Saving Data")
print("="*60)

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / 'X_train.npy', X_train_normalized)
np.save(output_dir / 'y_train.npy', y_train)
np.save(output_dir / 'X_test.npy', X_test_normalized)
np.save(output_dir / 'y_test.npy', y_test)

preprocessing_info = {
    'config': CONFIG,
    'mean': mean,
    'std': std,
    'activity_to_label': activity_to_label,
    'label_to_activity': label_to_activity,
    'train_users': train_users,
    'test_users': test_users,
    'train_shape': X_train_normalized.shape,
    'test_shape': X_test_normalized.shape,
}

with open(output_dir / 'preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print(f"✓ Saved to {output_dir}/")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)

print(f"""
Summary:
  ✓ Activities: {len(CONFIG['selected_activities'])}
  ✓ Total users: {len(all_users)}
  ✓ Train users: {len(train_users)}
  ✓ Test users: {len(test_users)}
  ✓ Window: {CONFIG['window_size']} samples ({CONFIG['window_size']/CONFIG['sampling_rate']:.2f}s)
  ✓ Overlap: {CONFIG['overlap']*100:.0f}%
  
  Training: {X_train_normalized.shape[0]:,} windows
  Test: {X_test_normalized.shape[0]:,} windows
""")

DATA PREPROCESSING PIPELINE

STEP 1: Loading Dataset
✓ Loaded 495,446 samples

STEP 2: Filtering Activities
✓ Filtered to 495,446 samples

Samples per activity:
  Eating              : 85,188
  Nail_Biting         : 111,189
  Face_Touch          : 84,420
  Smoking             : 94,586
  Staying_Still       : 120,063

STEP 3: Creating Train/Test Split
Total users found: 37

Train users (30): [1, 2, 3, 4, 5, 6, 7, 9, 10, 12, 13, 14, 16, 17, 18, 20, 22, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37]
Test users (7): [8, 11, 15, 19, 21, 23, 29]

Train samples: 403,936 (81.5%)
Test samples: 91,510 (18.5%)

STEP 4: Creating Label Mapping
Label mapping:
  0: Eating
  1: Nail_Biting
  2: Face_Touch
  3: Smoking
  4: Staying_Still

STEP 5: Creating Sliding Windows
✓ Created windows
  Training: 6,137 windows
  Test: 1,383 windows

STEP 6: Normalizing Data
✓ Data normalized

STEP 7: Class Distribution
Training set:
  0: Eating               -  1,098 (17.89%)
  1: Nail_Biting          -  1,311